# M2.Ex3: Enrichment

Add two enrichment steps to the Docling pipeline:

1. **Picture Classification** — labels each picture (chart type, flow diagram, logo, signature, etc.) using the `DocumentFigureClassifier` model.
2. **Picture Description (Captioning)** — generates a short caption for each picture using a vision model (SmolVLM by default).

Both steps are enabled together in a single pipeline, so one conversion gives us both results.

In [ ]:
# Install Docling if you don't have it yet
%pip install docling

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    smolvlm_picture_description,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import PictureItem

## 1) Configure the pipeline

Enable both enrichment steps at once.

In [ ]:
pipeline_options = PdfPipelineOptions()

# Common settings - both enrichments need access to the picture images
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2

# Task 1: picture classification
pipeline_options.do_picture_classification = True
 
# Task 2: picture description (captioning) using a small local VLM.
# Swap for `granite_picture_description` if you want a stronger (heavier) model.
pipeline_options.do_picture_description = True
pipeline_options.picture_description_options = smolvlm_picture_description

## 2) Build the converter using these options

In [ ]:
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)

## 3) Convert a sample PDF (same one used in the docs)

In [ ]:
result = converter.convert("https://arxiv.org/pdf/2501.17887")
doc = result.document

## 4) Print the results for each picture

Each picture stores enrichment results as annotations:
- `kind == "classification"` -> the predicted picture class with confidence
- `kind == "description"` -> the generated caption text

In [ ]:
print("Enrichment results per picture:")
print("=" * 60)

for i, (picture, _level) in enumerate(doc.iterate_items(), start=1):
    if not isinstance(picture, PictureItem):
        continue

    print(f"\nPicture #{i}")

    # --- Task 1: classification ---
    classifications = [
        ann for ann in picture.annotations
        if ann.kind == "classification"
    ]

    if classifications:
        for ann in classifications:
            top = ann.predicted_classes[0]  # best prediction
            print(f"  class:   {top.class_name} "
                  f"(confidence {top.confidence:.2f})")
    else:
        print("  class:   (no classification produced)")

    # --- Task 2: description ---
    descriptions = [
        ann for ann in picture.annotations
        if ann.kind == "description"
    ]

    if descriptions:
        for ann in descriptions:
            print(f"  caption: {ann.text}")
    else:
        print("  caption: (no caption produced)")